In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import os
import glob
from typing import Dict, Any, Optional
import plotly.graph_objects as go



# Cálculo de dz a partir well data


In [ ]:
# =========================
# CONFIGURACIÓN
# =========================
file_path = r"C:\Users\Mariana\Documents\freshwater_lens\data\rawdy\rawdy_1cm_lrs70\LRS70_D_YSI_20220816_processed.csv"
depth_col = "Vertical Position m"

# =========================
# LECTURA Y LIMPIEZA
# =========================
df = pd.read_csv(file_path)

df = df.dropna(subset=[depth_col])
df = df.sort_values(by=depth_col).reset_index(drop=True)

# =========================
# CÁLCULO DE Δz
# =========================
depth_m = df[depth_col].values

dz_m = np.diff(depth_m)

# Convertir a cm
depth_cm = depth_m * 100
dz_cm = dz_m * 100

depth_mid_cm = depth_cm[1:]

# =========================
# GRÁFICA
# =========================
plt.figure(figsize=(6, 8))

plt.scatter(depth_mid_cm, dz_cm, s=10)

plt.xlabel("Depth (cm)")
plt.ylabel("Δz between consecutive points (cm)")
plt.title("Vertical sampling spacing")

# Evitar notación científica
ax = plt.gca()
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())

# Forzar formato normal (no 1e-3)
ax.ticklabel_format(style='plain', axis='both')

plt.grid(True)

plt.show()

# =========================
# INFO EXTRA
# =========================
print("Resumen de espaciamiento (cm):")
print(f"Min dz: {dz_cm.min():.5f} cm")
print(f"Max dz: {dz_cm.max():.5f} cm")
print(f"Mean dz: {dz_cm.mean():.5f} cm")
print(f"Median dz: {np.median(dz_cm):.5f} cm")

# Distribución de datos a lo largo de un perfil en relación a punto medio y elbow sz


In [ ]:
def analyze_profile_data_balance(
    csv_path: str,
    dgh_center: float,
    dgh_sz_elbow: float,
    vp_column: str = 'Vertical Position m',
    sec_column: str = 'Corrected sp Cond [µS/cm]',
    sort_by_depth: bool = True
) -> Dict[str, Any]:
    """
    Analiza cómo se distribuyen los puntos medidos de un perfil de pozo
    respecto a dos valores de conductividad eléctrica:
        - dgh_center
        - dgh_sz_elbow

    Para cada valor de conductividad, encuentra la profundidad (Vertical Position m)
    cuyo valor de conductividad es el más cercano.

    Luego cuenta:
        1) Cuántos puntos están por encima y por debajo de la profundidad de dgh_center
        2) Cuántos puntos están entre la profundidad de dgh_center y la de dgh_sz_elbow
        3) Cuántos puntos están por debajo de la profundidad de dgh_sz_elbow

    Parameters
    ----------
    csv_path : str
        Ruta al archivo CSV del perfil.
    dgh_center : float
        Valor de conductividad para el centro DGH.
    dgh_sz_elbow : float
        Valor de conductividad para el elbow de la zona de transición.
    vp_column : str
        Nombre de la columna de profundidad.
    sec_column : str
        Nombre de la columna de conductividad.
    sort_by_depth : bool
        Si True, ordena el DataFrame por profundidad.

    Returns
    -------
    Dict[str, Any]
        Diccionario con profundidades estimadas, conteos y porcentajes.
    """

    # Leer archivo
    df = pd.read_csv(csv_path)

    # Validar columnas
    required_cols = [vp_column, sec_column]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas en el CSV: {missing_cols}")

    # Quedarnos solo con columnas necesarias y limpiar NaN
    df = df[[vp_column, sec_column]].copy()
    df[vp_column] = pd.to_numeric(df[vp_column], errors='coerce')
    df[sec_column] = pd.to_numeric(df[sec_column], errors='coerce')
    df = df.dropna(subset=[vp_column, sec_column])

    if df.empty:
        raise ValueError("No hay datos válidos después de limpiar NaN o valores no numéricos.")

    # Ordenar por profundidad
    if sort_by_depth:
        df = df.sort_values(by=vp_column).reset_index(drop=True)

    # Encontrar punto más cercano a dgh_center
    idx_center = (df[sec_column] - dgh_center).abs().idxmin()
    center_depth = df.loc[idx_center, vp_column]
    center_sec = df.loc[idx_center, sec_column]

    # Encontrar punto más cercano a dgh_sz_elbow
    idx_elbow = (df[sec_column] - dgh_sz_elbow).abs().idxmin()
    elbow_depth = df.loc[idx_elbow, vp_column]
    elbow_sec = df.loc[idx_elbow, sec_column]

    # Asegurar orden vertical entre ambos puntos
    upper_depth = min(center_depth, elbow_depth)
    lower_depth = max(center_depth, elbow_depth)

    # Conteos
    total_points = len(df)

    # Respecto a dgh_center
    n_shallower_than_center = (df[vp_column] < center_depth).sum()
    n_deeper_than_center = (df[vp_column] > center_depth).sum()
    n_equal_center_depth = (df[vp_column] == center_depth).sum()

    # Entre center y elbow
    n_between_center_and_elbow = ((df[vp_column] > upper_depth) & (df[vp_column] < lower_depth)).sum()

    # Más profundos que elbow
    n_deeper_than_elbow = (df[vp_column] > elbow_depth).sum()
    n_shallower_than_elbow = (df[vp_column] < elbow_depth).sum()
    n_equal_elbow_depth = (df[vp_column] == elbow_depth).sum()

    # Segmentación útil del perfil
    n_above_upper = (df[vp_column] < upper_depth).sum()
    n_between = ((df[vp_column] >= upper_depth) & (df[vp_column] <= lower_depth)).sum()
    n_below_lower = (df[vp_column] > lower_depth).sum()

    results = {
        "input_parameters": {
            "dgh_center": dgh_center,
            "dgh_sz_elbow": dgh_sz_elbow,
            "vp_column": vp_column,
            "sec_column": sec_column,
        },
        "matched_points": {
            "dgh_center_match": {
                "row_index": int(idx_center),
                "matched_depth": float(center_depth),
                "matched_conductivity": float(center_sec),
                "absolute_difference": float(abs(center_sec - dgh_center)),
            },
            "dgh_sz_elbow_match": {
                "row_index": int(idx_elbow),
                "matched_depth": float(elbow_depth),
                "matched_conductivity": float(elbow_sec),
                "absolute_difference": float(abs(elbow_sec - dgh_sz_elbow)),
            },
        },
        "counts_relative_to_center": {
            "shallower_than_center_depth": int(n_shallower_than_center),
            "deeper_than_center_depth": int(n_deeper_than_center),
            "exactly_at_center_depth": int(n_equal_center_depth),
        },
        "counts_relative_to_elbow": {
            "shallower_than_elbow_depth": int(n_shallower_than_elbow),
            "deeper_than_elbow_depth": int(n_deeper_than_elbow),
            "exactly_at_elbow_depth": int(n_equal_elbow_depth),
        },
        "counts_between_depths": {
            "between_center_and_elbow_exclusive": int(n_between_center_and_elbow),
        },
        "three_zone_distribution": {
            "above_upper_depth": int(n_above_upper),
            "between_upper_and_lower_inclusive": int(n_between),
            "below_lower_depth": int(n_below_lower),
            "upper_depth": float(upper_depth),
            "lower_depth": float(lower_depth),
        },
        "percentages_three_zone_distribution": {
            "above_upper_depth_pct": float(n_above_upper / total_points * 100),
            "between_upper_and_lower_inclusive_pct": float(n_between / total_points * 100),
            "below_lower_depth_pct": float(n_below_lower / total_points * 100),
        },
        "total_points": int(total_points),
    }

    return results


def print_profile_balance_results(results: Dict[str, Any]) -> None:
    """Imprime los resultados de forma amigable."""
    
    center = results["matched_points"]["dgh_center_match"]
    elbow = results["matched_points"]["dgh_sz_elbow_match"]
    center_counts = results["counts_relative_to_center"]
    elbow_counts = results["counts_relative_to_elbow"]
    between_counts = results["counts_between_depths"]
    zones = results["three_zone_distribution"]
    pct = results["percentages_three_zone_distribution"]

    print("\n" + "=" * 60)
    print("BALANCE DE DATOS DEL PERFIL DE POZO")
    print("=" * 60)

    print("\n[1] Punto más cercano a dgh_center")
    print(f"  Profundidad encontrada:    {center['matched_depth']:.4f} m")
    print(f"  Conductividad encontrada:  {center['matched_conductivity']:.2f} µS/cm")
    print(f"  Diferencia absoluta:       {center['absolute_difference']:.2f} µS/cm")

    print("\n[2] Punto más cercano a dgh_sz_elbow")
    print(f"  Profundidad encontrada:    {elbow['matched_depth']:.4f} m")
    print(f"  Conductividad encontrada:  {elbow['matched_conductivity']:.2f} µS/cm")
    print(f"  Diferencia absoluta:       {elbow['absolute_difference']:.2f} µS/cm")

    print("\n[3] Conteos respecto a la profundidad de dgh_center")
    print(f"  Puntos menos profundos:    {center_counts['shallower_than_center_depth']}")
    print(f"  Puntos más profundos:      {center_counts['deeper_than_center_depth']}")
    print(f"  Puntos exactamente ahí:    {center_counts['exactly_at_center_depth']}")

    print("\n[4] Conteos entre dgh_center y dgh_sz_elbow")
    print(f"  Puntos entre ambas profundidades (exclusivo): {between_counts['between_center_and_elbow_exclusive']}")

    print("\n[5] Conteos respecto a la profundidad de dgh_sz_elbow")
    print(f"  Puntos menos profundos:    {elbow_counts['shallower_than_elbow_depth']}")
    print(f"  Puntos más profundos:      {elbow_counts['deeper_than_elbow_depth']}")
    print(f"  Puntos exactamente ahí:    {elbow_counts['exactly_at_elbow_depth']}")

    print("\n[6] Distribución en tres zonas")
    print(f"  Arriba del punto más somero:      {zones['above_upper_depth']} ({pct['above_upper_depth_pct']:.2f}%)")
    print(f"  Entre ambos puntos:               {zones['between_upper_and_lower_inclusive']} ({pct['between_upper_and_lower_inclusive_pct']:.2f}%)")
    print(f"  Debajo del punto más profundo:    {zones['below_lower_depth']} ({pct['below_lower_depth_pct']:.2f}%)")

    print(f"\nTotal de puntos: {results['total_points']}")
    print("=" * 60)

In [ ]:
csv_path = r"C:\Users\Mariana\Documents\freshwater_lens\data\rawdy\rawdy_sat_51w2p_1cm_lrs70\LRS70_D_YSI_20220131_processed.csv"

results = analyze_profile_data_balance(
    csv_path=csv_path,
    dgh_center=27800,
    dgh_sz_elbow=52660,
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]'
)

print_profile_balance_results(results)